In [ ]:
import pandas as pd
from pathlib import Path

IN_CHUNKS = Path(r"E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\preprocess_result\chunking_result\sirah_chunks_final.csv")
OUT_SEED  = Path(r"E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\preprocess_result\chunking_result\sirah_manual_seed.csv")

dfc = pd.read_csv(IN_CHUNKS, sep=";", encoding="utf-8-sig").fillna("")
dfc.columns = dfc.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()

print("Rows:", len(dfc))
print("Cols:", dfc.columns.tolist())
dfc.head(3)


Rows: 1099
Cols: ['chunk_id', 'doc_id', 'chunk_index', 'judul_bab', 'judul_sub_bab', 'halaman', 'teks_chunk']


,chunk_id,doc_id,chunk_index,judul_bab,judul_sub_bab,halaman,teks_chunk
0,000000-001,0,1,POSISI BANGSA ARAB DAN KAUMNYA,UNLABELED SECTION,34,Pada hakikatnya istilah Sirah Nabawiyah merupa...
1,000001-001,1,1,POSISI BANGSA ARAB DAN KAUMNYA,Posisi Bangsa Arab,34-35,"Menurut bahasa, Arab artinya padang pasir, tan..."
2,000001-002,1,2,POSISI BANGSA ARAB DAN KAUMNYA,Posisi Bangsa Arab,34-35,Sekalipun begitu mereka tetap hidup berdamping...


In [ ]:
PER_BAB = 25        # ambil maksimal 25 chunk per BAB
RANDOM_STATE = 42   # biar hasilnya konsisten

seed = (
    dfc.groupby("judul_bab", group_keys=False)
       .apply(lambda g: g.sample(n=min(len(g), PER_BAB), random_state=RANDOM_STATE))
       .reset_index(drop=True)
)

print("Seed size:", len(seed))
seed[["judul_bab"]].value_counts().head(10)


Seed size: 847


C:\Users\Owner\AppData\Local\Temp\ipykernel_20724\1661329492.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(len(g), PER_BAB), random_state=RANDOM_STATE))


judul_bab                                             
KORESPONDENSI DENGAN BEBERAPA RAJA DAN AMIR               25
MANUSIA MEMASUKI AGAMA ALLAH SECARA BERBONDONG-BONDONG    25
KELAHIRAN DAN EMPAT PULUH TAHUN SEBELUM NUBUWAH           25
KONSPIRASI UNTUK MEMBUNUH NABI                            25
PERANG UHUD                                               25
PERJANJIAN HUDAIBIYAH                                     25
PERANG KHAIBAR DAN WADIL QURA                             25
PERANG DAN PENAKLUKAN MAKKAH                              25
PERANG HUNAIN                                             25
PERANG AHZAB ATAU KHANDAQ                                 25
Name: count, dtype: int64

In [ ]:
# kolom inti + konteks (konteks opsional, tapi membantu manusia)
cols = ["chunk_id", "doc_id", "chunk_index", "judul_bab", "judul_sub_bab", "halaman", "teks_chunk"]

seed_out = seed[cols].copy()

# kolom anotasi manual
seed_out["entity_text"] = ""     # isi teks entitas persis seperti di teks_chunk
seed_out["label"] = ""           # PERSON / LOCATION / EVENT / TIME
seed_out["notes"] = ""           # opsional catatan

# opsional: kolom offset (boleh dikosongi dulu)
seed_out["start_char"] = ""      # indeks awal (opsional)
seed_out["end_char"] = ""        # indeks akhir (opsional)

seed_out.to_csv(OUT_SEED, index=False, sep=";", encoding="utf-8-sig")
print("Saved:", OUT_SEED)
seed_out.head(5)


Saved: E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\preprocess_result\chunking_result\sirah_manual_seed.csv


,chunk_id,doc_id,chunk_index,judul_bab,judul_sub_bab,halaman,teks_chunk,entity_text,label,notes,start_char,end_char
0,000354-001,354,1,ABU BAKAR MENUNAIKAN HAJI,UNLABELED SECTION,572,Pada bulan Dzul Qi'dah atau Dzul Hijjah tahu 9...,,,,,
1,000010-001,10,1,AGAMA BANGSA ARAB,UNLABELED SECTION,56-64,Mayoritas bangsa Arab mengikuti dakwah Isma’il...,,,,,
2,000010-002,10,2,AGAMA BANGSA ARAB,UNLABELED SECTION,56-64,Inilah tiga berhala yang paling besar. Setelah...,,,,,
3,000010-006,10,6,AGAMA BANGSA ARAB,UNLABELED SECTION,56-64,Akan tetapi orang-orang kafir membuat-buat ked...,,,,,
4,000011-001,11,1,AGAMA BANGSA ARAB,Kondisi Kehidupan Agama,64-65,Itulah agama-agama yang ada pada saat kedatang...,,,,,
